In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD
# ============================================================
h_train = pd.read_csv("/kaggle/input/datasets/mirceafinalboss/dldataset/h_train.csv")
m_train = pd.read_csv("/kaggle/input/datasets/mirceafinalboss/dldataset/m_train.csv")
p_train = pd.read_csv("/kaggle/input/datasets/mirceafinalboss/dldataset/p_train.csv")

print("P — model dist:")
print(p_train['model'].value_counts())
print("\nP — attack dist:")
print(p_train['attack'].value_counts())

# ============================================================
# SAMPLING STRATIFICAT (domain + model) — 10k per clasa
# ============================================================

N = 50_000

def stratified_sample(df, group_cols, n, seed=42):
    return (
        df.groupby(group_cols, group_keys=False)
        .apply(lambda x: x.sample(
            n=min(len(x), max(1, int(n * len(x) / len(df)))),
            random_state=seed
        ), include_groups=False)
        .sample(frac=1, random_state=seed)
        .head(n)
        .reset_index(drop=True)
    )

h_sample = stratified_sample(h_train, ['domain'], N)
m_sample = stratified_sample(m_train, ['domain', 'model'], N)
p_sample = stratified_sample(p_train, ['domain', 'model'], N)

h_sample['label'] = 1
m_sample['label'] = 0
p_sample['label'] = 0

# Exp 1: H + M
train_exp1 = pd.concat([h_sample, m_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Exp 2: H + M + P
train_exp2 = pd.concat([h_sample, m_sample, p_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Exp1: {len(train_exp1)} | Exp2: {len(train_exp2)}")
print(f"\nExp1 label dist:\n{train_exp1['label'].value_counts()}")
print(f"\nExp2 label dist:\n{train_exp2['label'].value_counts()}")

train_exp1.to_csv('/kaggle/working/train_exp1.csv', index=False)
train_exp2.to_csv('/kaggle/working/train_exp2.csv', index=False)
print("\nSalvat.")

P — model dist:
model
mistral-chat    8683
llama-chat      8680
mpt-chat        8673
mistral         8671
gpt2            8669
mpt             8653
gpt3            4378
cohere          4376
chatgpt         4376
cohere-chat     4376
gpt4            4373
Name: count, dtype: int64

P — attack dist:
attack
paraphrase    73908
Name: count, dtype: int64
Exp1: 99969 | Exp2: 149938

Exp1 label dist:
label
1    49998
0    49971
Name: count, dtype: int64

Exp2 label dist:
label
0    99940
1    49998
Name: count, dtype: int64

Salvat.


In [2]:
!pip install transformers datasets scikit-learn -q

In [3]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_scheduler
from torch.optim import AdamW
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    precision_score, recall_score, confusion_matrix,
    matthews_corrcoef, average_precision_score,
    classification_report
)
from torch.cuda.amp import autocast, GradScaler
import time
import json
from huggingface_hub import HfApi, login

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")


Device: cuda
GPU: Tesla T4


In [ ]:

from huggingface_hub import HfApi, login, create_repo

login(token="")

HF_REPO = "Mosshato/ai-text-detector"  # schimba doar username-ul

# Creeaza repo daca nu exista
create_repo(HF_REPO, exist_ok=True, private=False)
print(f"Repo ready: https://huggingface.co/{HF_REPO}")

Repo ready: https://huggingface.co/Mosshato/ai-text-detector


In [5]:
train_df = pd.read_csv("/kaggle/working/train_exp1.csv")
h_val = pd.read_csv("/kaggle/input/datasets/mirceafinalboss/dldataset/h_val.csv")
m_val = pd.read_csv("/kaggle/input/datasets/mirceafinalboss/dldataset/m_val.csv")

# Val set — sample mic pentru viteza
h_val_sample = h_val.sample(n=2000, random_state=42)
m_val_sample = m_val.sample(n=2000, random_state=42)

h_val_sample['label'] = 1
m_val_sample['label'] = 0

val_df = pd.concat([h_val_sample, m_val_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")
print(f"Train label dist:\n{train_df['label'].value_counts()}")

Train: 99969 | Val: 4000
Train label dist:
label
1    49998
0    49971
Name: count, dtype: int64


In [6]:
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts  = df['generation'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = TextDataset(train_df, tokenizer)
val_dataset   = TextDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 3125 | Val batches: 63


In [7]:
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

# NU mai ai niciun freeze — toți parametrii trainable
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parametri:  {total:,}")
print(f"Trainable:        {trainable:,}  (100%)")

model = model.to(device)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parametri:  124,647,170
Trainable:        124,647,170  (100%)


In [8]:
EPOCHS       = 3
LR           = 1e-5
WARMUP_RATIO = 0.06

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.01
)

scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.amp.GradScaler('cuda')

print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")


Total steps: 9375 | Warmup steps: 562


In [9]:
def evaluate(model, loader):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            probs = torch.softmax(outputs.logits, dim=-1)[:, 1]  # p_human
            preds = outputs.logits.argmax(dim=-1)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    auroc = roc_auc_score(all_labels, all_probs)
    f1    = f1_score(all_labels, all_preds, average='macro')
    acc   = accuracy_score(all_labels, all_preds)
    return auroc, f1, acc

best_auroc = 0
history    = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        if step % 100 == 0:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | Elapsed: {elapsed:.0f}s")

    avg_loss = total_loss / len(train_loader)
    auroc, f1, acc = evaluate(model, val_loader)

    history.append({'epoch': epoch+1, 'loss': avg_loss, 'auroc': auroc, 'f1': f1, 'acc': acc})

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch+1} COMPLETE")
    print(f"  Train Loss: {avg_loss:.4f}")
    print(f"  Val AUROC:  {auroc:.4f}")
    print(f"  Val F1:     {f1:.4f}")
    print(f"  Val Acc:    {acc:.4f}")
    print(f"  Timp:       {(time.time()-t0)/60:.1f} min")
    print(f"{'='*60}\n")

    # Save best model
    if auroc > best_auroc:
        best_auroc = auroc
        model.save_pretrained('/kaggle/working/best_model')
        tokenizer.save_pretrained('/kaggle/working/best_model')
        print(f"  ✓ Best model salvat (AUROC: {auroc:.4f})")

print(f"\nTraining complet! Best AUROC: {best_auroc:.4f}")
print("\nHistory:")
print(pd.DataFrame(history).to_string(index=False))

/tmp/ipykernel_23/3596753362.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 1 | Step 0/3125 | Loss: 0.6880 | Elapsed: 2s
  Epoch 1 | Step 100/3125 | Loss: 0.6936 | Elapsed: 38s
  Epoch 1 | Step 200/3125 | Loss: 0.4648 | Elapsed: 75s
  Epoch 1 | Step 300/3125 | Loss: 0.2674 | Elapsed: 114s
  Epoch 1 | Step 400/3125 | Loss: 0.1128 | Elapsed: 151s
  Epoch 1 | Step 500/3125 | Loss: 0.0834 | Elapsed: 189s
  Epoch 1 | Step 600/3125 | Loss: 0.0977 | Elapsed: 227s
  Epoch 1 | Step 700/3125 | Loss: 0.0450 | Elapsed: 264s
  Epoch 1 | Step 800/3125 | Loss: 0.1583 | Elapsed: 302s
  Epoch 1 | Step 900/3125 | Loss: 0.0043 | Elapsed: 340s
  Epoch 1 | Step 1000/3125 | Loss: 0.0068 | Elapsed: 377s
  Epoch 1 | Step 1100/3125 | Loss: 0.0020 | Elapsed: 415s
  Epoch 1 | Step 1200/3125 | Loss: 0.0118 | Elapsed: 452s
  Epoch 1 | Step 1300/3125 | Loss: 0.1540 | Elapsed: 490s
  Epoch 1 | Step 1400/3125 | Loss: 0.0559 | Elapsed: 527s
  Epoch 1 | Step 1500/3125 | Loss: 0.0130 | Elapsed: 565s
  Epoch 1 | Step 1600/3125 | Loss: 0.0022 | Elapsed: 602s
  Epoch 1 | Step 1700/3125 | L

/tmp/ipykernel_23/3596753362.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



EPOCH 1 COMPLETE
  Train Loss: 0.0989
  Val AUROC:  0.9984
  Val F1:     0.9615
  Val Acc:    0.9615
  Timp:       19.8 min



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  Epoch 3 | Step 0/3125 | Loss: 0.0008 | Elapsed: 1s
  Epoch 3 | Step 100/3125 | Loss: 0.0003 | Elapsed: 38s
  Epoch 3 | Step 200/3125 | Loss: 0.0001 | Elapsed: 75s
  Epoch 3 | Step 300/3125 | Loss: 0.0004 | Elapsed: 112s
  Epoch 3 | Step 400/3125 | Loss: 0.0002 | Elapsed: 150s
  Epoch 3 | Step 500/3125 | Loss: 0.0003 | Elapsed: 187s
  Epoch 3 | Step 600/3125 | Loss: 0.0001 | Elapsed: 224s
  Epoch 3 | Step 700/3125 | Loss: 0.0001 | Elapsed: 262s
  Epoch 3 | Step 800/3125 | Loss: 0.0207 | Elapsed: 299s
  Epoch 3 | Step 900/3125 | Loss: 0.0001 | Elapsed: 336s
  Epoch 3 | Step 1000/3125 | Loss: 0.0006 | Elapsed: 373s
  Epoch 3 | Step 1100/3125 | Loss: 0.0014 | Elapsed: 411s
  Epoch 3 | Step 1200/3125 | Loss: 0.0002 | Elapsed: 448s
  Epoch 3 | Step 1300/3125 | Loss: 0.0021 | Elapsed: 485s
  Epoch 3 | Step 1400/3125 | Loss: 0.0001 | Elapsed: 523s
  Epoch 3 | Step 1500/3125 | Loss: 0.0005 | Elapsed: 560s
  Epoch 3 | Step 1600/3125 | Loss: 0.0011 | Elapsed: 597s
  Epoch 3 | Step 1700/3125 | L

/tmp/ipykernel_23/3596753362.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



EPOCH 3 COMPLETE
  Train Loss: 0.0101
  Val AUROC:  0.9993
  Val F1:     0.9687
  Val Acc:    0.9688
  Timp:       19.6 min



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model salvat (AUROC: 0.9993)

Training complet! Best AUROC: 0.9993

History:
 epoch     loss    auroc       f1     acc
     1 0.098949 0.998363 0.961451 0.96150
     2 0.023338 0.999082 0.970480 0.97050
     3 0.010097 0.999318 0.968724 0.96875


In [10]:
def evaluate(model, loader, split_name="val"):
    model.eval()
    all_labels, all_probs, all_preds = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            probs = torch.softmax(outputs.logits, dim=-1)[:, 1]  # p_human
            preds = outputs.logits.argmax(dim=-1)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    all_preds  = np.array(all_preds)

    # ── Metrici ──────────────────────────────────────────────
    auroc    = roc_auc_score(all_labels, all_probs)
    auprc    = average_precision_score(all_labels, all_probs)  # Area Under Precision-Recall Curve
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_bin   = f1_score(all_labels, all_preds, average='binary')
    acc      = accuracy_score(all_labels, all_preds)
    prec     = precision_score(all_labels, all_preds, zero_division=0)
    rec      = recall_score(all_labels, all_preds, zero_division=0)
    mcc      = matthews_corrcoef(all_labels, all_preds)  # Matthew's Correlation Coefficient
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # True Negative Rate

    metrics = {
        'split':       split_name,
        'auroc':       round(auroc, 4),
        'auprc':       round(auprc, 4),       # util când clasele sunt dezechilibrate
        'f1_macro':    round(f1_macro, 4),
        'f1_binary':   round(f1_bin, 4),
        'accuracy':    round(acc, 4),
        'precision':   round(prec, 4),        # din ce zice AI, cat e corect
        'recall':      round(rec, 4),         # din tot ce e human, cat prinde
        'specificity': round(specificity, 4), # din tot ce e AI, cat prinde
        'mcc':         round(mcc, 4),         # cel mai echilibrat scor general
        'tp': int(tp), 'tn': int(tn),
        'fp': int(fp), 'fn': int(fn)
    }

    # ── Print frumos ─────────────────────────────────────────
    print(f"\n  {'─'*45}")
    print(f"  METRICI [{split_name.upper()}]")
    print(f"  {'─'*45}")
    print(f"  AUROC:       {auroc:.4f}  ← principal")
    print(f"  AUPRC:       {auprc:.4f}")
    print(f"  MCC:         {mcc:.4f}  ← cel mai echilibrat")
    print(f"  F1 Macro:    {f1_macro:.4f}")
    print(f"  F1 Binary:   {f1_bin:.4f}")
    print(f"  Accuracy:    {acc:.4f}")
    print(f"  Precision:   {prec:.4f}")
    print(f"  Recall:      {rec:.4f}")
    print(f"  Specificity: {specificity:.4f}")
    print(f"  {'─'*45}")
    print(f"  Confusion Matrix:")
    print(f"              Pred AI  Pred Human")
    print(f"  Real AI       {tn:5d}      {fp:5d}   (TN, FP)")
    print(f"  Real Human    {fn:5d}      {tp:5d}   (FN, TP)")
    print(f"  {'─'*45}\n")

    return metrics


In [11]:
best_auroc = 0
history    = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        if step % 100 == 0:
            elapsed = time.time() - t0
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | Elapsed: {elapsed:.0f}s")

    avg_loss = total_loss / len(train_loader)
    metrics  = evaluate(model, val_loader, split_name=f"val_epoch{epoch+1}")
    metrics['train_loss'] = round(avg_loss, 4)
    metrics['epoch']      = epoch + 1
    metrics['time_min']   = round((time.time() - t0) / 60, 1)

    history.append(metrics)

    print(f"  Timp total epoch: {metrics['time_min']} min")

    # ── Save best ────────────────────────────────────────────
    if metrics['auroc'] > best_auroc:
        best_auroc = metrics['auroc']
        model.save_pretrained('/kaggle/working/best_model')
        tokenizer.save_pretrained('/kaggle/working/best_model')

        # Salveaza si metricile
        with open('/kaggle/working/best_model/metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)

        print(f"  ✓ Best model salvat local (AUROC: {metrics['auroc']:.4f})")

        # ── Push to HuggingFace ──────────────────────────────
        model.push_to_hub(HF_REPO, commit_message=f"Epoch {epoch+1} — AUROC {metrics['auroc']:.4f}")
        tokenizer.push_to_hub(HF_REPO, commit_message=f"Tokenizer epoch {epoch+1}")
        print(f"  ✓ Pushuit pe HuggingFace: {HF_REPO}")

print(f"\nTraining complet! Best AUROC: {best_auroc:.4f}")
print("\n=== HISTORY COMPLET ===")
print(pd.DataFrame(history)[['epoch', 'train_loss', 'auroc', 'auprc', 'mcc', 'f1_macro', 'accuracy', 'precision', 'recall', 'time_min']].to_string(index=False))

  Epoch 1 | Step 0/3125 | Loss: 0.0001 | Elapsed: 1s
  Epoch 1 | Step 100/3125 | Loss: 0.0003 | Elapsed: 38s
  Epoch 1 | Step 200/3125 | Loss: 0.0001 | Elapsed: 75s
  Epoch 1 | Step 300/3125 | Loss: 0.0001 | Elapsed: 112s
  Epoch 1 | Step 400/3125 | Loss: 0.0001 | Elapsed: 149s
  Epoch 1 | Step 500/3125 | Loss: 0.0002 | Elapsed: 187s
  Epoch 1 | Step 600/3125 | Loss: 0.0002 | Elapsed: 224s
  Epoch 1 | Step 700/3125 | Loss: 0.0001 | Elapsed: 261s
  Epoch 1 | Step 800/3125 | Loss: 0.0002 | Elapsed: 298s
  Epoch 1 | Step 900/3125 | Loss: 0.0002 | Elapsed: 336s
  Epoch 1 | Step 1000/3125 | Loss: 0.0002 | Elapsed: 373s
  Epoch 1 | Step 1100/3125 | Loss: 0.0001 | Elapsed: 410s
  Epoch 1 | Step 1200/3125 | Loss: 0.0002 | Elapsed: 447s
  Epoch 1 | Step 1300/3125 | Loss: 0.0003 | Elapsed: 485s
  Epoch 1 | Step 1400/3125 | Loss: 0.0001 | Elapsed: 522s
  Epoch 1 | Step 1500/3125 | Loss: 0.0002 | Elapsed: 559s
  Epoch 1 | Step 1600/3125 | Loss: 0.0002 | Elapsed: 597s
  Epoch 1 | Step 1700/3125 | L

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Best model salvat local (AUROC: 0.9993)


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


  ✓ Pushuit pe HuggingFace: Mosshato/ai-text-detector
  Epoch 2 | Step 0/3125 | Loss: 0.0001 | Elapsed: 1s
  Epoch 2 | Step 100/3125 | Loss: 0.0001 | Elapsed: 38s
  Epoch 2 | Step 200/3125 | Loss: 0.0002 | Elapsed: 75s
  Epoch 2 | Step 300/3125 | Loss: 0.0001 | Elapsed: 113s
  Epoch 2 | Step 400/3125 | Loss: 0.0001 | Elapsed: 150s
  Epoch 2 | Step 500/3125 | Loss: 0.0001 | Elapsed: 187s
  Epoch 2 | Step 600/3125 | Loss: 0.0001 | Elapsed: 224s
  Epoch 2 | Step 700/3125 | Loss: 0.0003 | Elapsed: 262s
  Epoch 2 | Step 800/3125 | Loss: 0.0004 | Elapsed: 299s
  Epoch 2 | Step 900/3125 | Loss: 0.0001 | Elapsed: 336s
  Epoch 2 | Step 1000/3125 | Loss: 0.0001 | Elapsed: 373s
  Epoch 2 | Step 1100/3125 | Loss: 0.0001 | Elapsed: 411s
  Epoch 2 | Step 1200/3125 | Loss: 0.0001 | Elapsed: 448s
  Epoch 2 | Step 1300/3125 | Loss: 0.0001 | Elapsed: 485s
  Epoch 2 | Step 1400/3125 | Loss: 0.0002 | Elapsed: 522s
  Epoch 2 | Step 1500/3125 | Loss: 0.0002 | Elapsed: 560s
  Epoch 2 | Step 1600/3125 | Loss: